In [1]:
pip install scikit-optimize

Note: you may need to restart the kernel to use updated packages.


In [50]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dropout
from tensorflow.keras.regularizers import l2

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf

# scikit-optimize imports for Bayesian tuning
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.linear_model import LinearRegression

import matplotlib.pyplot as plt

In [23]:
# 2. NumPy
np.random.seed(42)

# 3. TensorFlow
tf.random.set_seed(42)

In [12]:
# Load data
df = pd.read_csv("./processed_plays.csv")

# Train/Validation/Test split (60/20/20)
train_val, test = train_test_split(df, test_size=0.20, random_state=42)
train, val      = train_test_split(train_val, test_size=0.20, random_state=42)

X_train, y_train = train.drop('yardsGained', axis=1), train['yardsGained']
X_val,   y_val   = val.drop('yardsGained', axis=1),   val['yardsGained']
X_test,  y_test  = test.drop('yardsGained', axis=1),  test['yardsGained']

In [37]:
df.head()

,down,yardsToGo,absoluteYardlineNumber,quarter,playAction,qbSneak,pff_runPassOption,yardsGained,secondsRemainingInQuarter,offFormation_EMPTY,...,passCoverage_Cover_3_Seam,passCoverage_Cover_6_Right,passCoverage_Goal_Line,passCoverage_Miscellaneous,passCoverage_Prevent,passCoverage_Quarters,passCoverage_Red_Zone,manZone_Man,manZone_Other,manZone_Zone
0,1,10,21,3,0,0,0,9,114,1,...,0,0,0,0,0,0,0,0,0,1
1,1,10,8,4,0,0,0,4,133,1,...,0,0,0,0,0,1,0,0,0,1
2,3,12,20,4,0,0,0,6,120,0,...,0,0,0,0,0,1,0,0,0,1
3,2,10,23,1,0,0,0,4,568,0,...,0,0,0,0,0,1,0,0,0,1
4,2,8,27,3,1,0,0,-1,136,0,...,0,0,0,0,0,0,0,1,0,0


In [51]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Define model architecture
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.2),
    Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.2),
    Dense(1)
])


In [52]:
# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',  # Mean Squared Error
    metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse'),
             tf.keras.metrics.MeanAbsoluteError(name='mae')]
)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate on train and validation sets
train_metrics = model.evaluate(X_train_scaled, y_train, verbose=0)
val_metrics   = model.evaluate(X_val_scaled, y_val, verbose=0)

print(f"\nNeural Net on train data → RMSE: {train_metrics[1]:.3f}  MAE: {train_metrics[2]:.3f}")
print(f"Neural Net on validation data → RMSE: {val_metrics[1]:.3f}  MAE: {val_metrics[2]:.3f}")

Epoch 1/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 89.3645 - mae: 6.1033 - rmse: 9.4438 - val_loss: 67.4084 - val_mae: 5.5205 - val_rmse: 8.2030
Epoch 2/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 77.2245 - mae: 5.8179 - rmse: 8.7766 - val_loss: 67.2176 - val_mae: 5.5465 - val_rmse: 8.1914
Epoch 3/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 75.9932 - mae: 5.8023 - rmse: 8.7057 - val_loss: 67.2135 - val_mae: 5.5339 - val_rmse: 8.1911
Epoch 4/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 76.0765 - mae: 5.7685 - rmse: 8.7107 - val_loss: 67.1068 - val_mae: 5.5462 - val_rmse: 8.1846
Epoch 5/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 75.7542 - mae: 5.7726 - rmse: 8.6921 - val_loss: 67.0709 - val_mae: 5.5506 - val_rmse: 8.1824
Epoch 6/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 75.2744 - mae: 5.7581 - rmse: 8.6645 - val_loss: 67.0735 - val_mae: 5.5508 - val_rmse: 8.1826
Epoch 7/100
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 74.7373 -

In [53]:
def evaluate(name, actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae  = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    print(f"{name} → RMSE: {rmse:.3f}, MAE: {mae:.3f}, R²: {r2:.3f}")

evaluate('Train', y_train, model.predict(X_train))
evaluate('Validation', y_val,   model.predict(X_val))
evaluate('Test', y_test,  model.predict(X_test))


319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 645us/step
Train → RMSE: 281.222, MAE: 239.339, R²: -1000.650
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 665us/step
Validation → RMSE: 283.144, MAE: 240.252, R²: -1168.251
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 627us/step
Test → RMSE: 281.303, MAE: 239.793, R²: -935.874
